# Verify cross-account Kaggle Dataset ownership

This creates a tiny **private** Dataset to verify which account owns an
upload. It is safe to run from a notebook hosted by account B while
authenticating with an API token belonging to account A.

Recommended Secrets:

- `KAGGLE_API_TOKEN`: current API token generated by the target owner.
- `KAGGLE_DATASET_OWNER`: public Kaggle username that must own the Dataset.

Legacy `KAGGLE_USERNAME` plus `KAGGLE_KEY` is also supported. The `key`
inside a legacy `kaggle.json` is **not** a `KAGGLE_API_TOKEN`; configure
it using the legacy pair instead. Never put a token directly in a cell
or print it. Revoke a temporarily shared token after this test.


In [ ]:
# Kaggle CLI is installed after selecting the authentication mode below.


## Load the target owner's credentials from Kaggle Secrets


In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def optional_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        return None
    value = value.strip() if value else ""
    return value or None

api_token = optional_secret("KAGGLE_API_TOKEN")
legacy_username = optional_secret("KAGGLE_USERNAME")
legacy_key = optional_secret("KAGGLE_KEY")
dataset_owner = optional_secret("KAGGLE_DATASET_OWNER") or legacy_username

print({
    "KAGGLE_API_TOKEN_present": bool(api_token),
    "KAGGLE_USERNAME_present": bool(legacy_username),
    "KAGGLE_KEY_present": bool(legacy_key),
    "secret_values_printed": False,
})
if bool(legacy_username) != bool(legacy_key):
    raise RuntimeError(
        "Legacy authentication is incomplete: KAGGLE_USERNAME and "
        "KAGGLE_KEY must both be present"
    )

if not dataset_owner:
    raise RuntimeError(
        "Add KAGGLE_DATASET_OWNER (recommended with KAGGLE_API_TOKEN) "
        "or KAGGLE_USERNAME to Kaggle Secrets"
    )
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{1,49}", dataset_owner):
    raise ValueError(f"Invalid KAGGLE_DATASET_OWNER: {dataset_owner!r}")

# Isolate the CLI from any OAuth/config file belonging to the account
# that hosts this notebook. Only the explicitly supplied Secret below
# is allowed to authenticate this test.
auth_config_dir = Path("/kaggle/working/typepro_owner_test_auth")
auth_config_dir.mkdir(parents=True, exist_ok=True)
os.environ["KAGGLE_CONFIG_DIR"] = str(auth_config_dir)

# A complete legacy pair is explicit and takes precedence. This is
# important in a cross-account Kaggle notebook because the hosting
# account may also expose its own access token/OAuth credentials.
if legacy_username and legacy_key:
    if dataset_owner.casefold() != legacy_username.casefold():
        raise RuntimeError(
            "With legacy credentials, KAGGLE_DATASET_OWNER must equal KAGGLE_USERNAME"
        )
    os.environ.pop("KAGGLE_API_TOKEN", None)
    os.environ["KAGGLE_USERNAME"] = legacy_username
    os.environ["KAGGLE_KEY"] = legacy_key
    auth_mode = "legacy KAGGLE_USERNAME/KAGGLE_KEY"
elif api_token:
    # Explicitly prefer the supplied target-owner token over any
    # credentials inherited from the notebook-hosting account.
    os.environ["KAGGLE_API_TOKEN"] = api_token
    os.environ.pop("KAGGLE_USERNAME", None)
    os.environ.pop("KAGGLE_KEY", None)
    auth_mode = "KAGGLE_API_TOKEN"
else:
    raise RuntimeError(
        "Add either KAGGLE_API_TOKEN, or both KAGGLE_USERNAME and KAGGLE_KEY"
    )

if auth_mode == "legacy KAGGLE_USERNAME/KAGGLE_KEY":
    # Kaggle CLI >=1.8 can consume the notebook-hosting account's
    # automatically provided access token before checking the explicit
    # legacy pair. Version 1.7.4.2 predates that authentication path.
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "kaggle==1.7.4.2",
        ],
        check=True,
    )
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
        check=True,
    )

# Import Kaggle in a fresh process. Importing authenticates immediately,
# so this reveals both the effective account and whether the supplied
# Secret was accepted instead of silently falling back to host OAuth.
identity_code = "\n".join([
    "import json",
    "import kaggle",
    "api = kaggle.api",
    'values = getattr(api, "config_values", {})',
    'username = values.get("username")',
    'method = values.get("auth_method") or "LEGACY_API_KEY"',
    'print("TYPEPRO_KAGGLE_IDENTITY=" + json.dumps({',
    '    "username": username,',
    '    "auth_method": method,',
    '}))',
])
identity_result = subprocess.run(
    [sys.executable, "-c", identity_code],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
identity_prefix = "TYPEPRO_KAGGLE_IDENTITY="
identity_line = next(
    (
        line for line in (identity_result.stdout or "").splitlines()
        if line.startswith(identity_prefix)
    ),
    None,
)
if identity_result.returncode or identity_line is None:
    raise RuntimeError(
        "Kaggle authentication preflight failed. If you copied the key from "
        "kaggle.json, use KAGGLE_USERNAME + KAGGLE_KEY instead of "
        f"KAGGLE_API_TOKEN. CLI output: {identity_result.stdout}"
    )
identity = json.loads(identity_line[len(identity_prefix):])
authenticated_username = (identity.get("username") or "").strip()
authenticated_method = (identity.get("auth_method") or "").strip()
expected_method = (
    "LEGACY_API_KEY"
    if auth_mode == "legacy KAGGLE_USERNAME/KAGGLE_KEY"
    else "ACCESS_TOKEN"
)
if authenticated_username.casefold() != dataset_owner.casefold():
    raise RuntimeError(
        f"Credential authenticated as {authenticated_username!r}, but the requested "
        f"Dataset owner is {dataset_owner!r}. Use a credential generated by "
        "the requested owner account."
    )
if authenticated_method != expected_method:
    raise RuntimeError(
        f"The supplied {auth_mode} was not accepted; Kaggle CLI used "
        f"{authenticated_method!r} instead. If this is a key from kaggle.json, "
        "remove KAGGLE_API_TOKEN and set KAGGLE_USERNAME + KAGGLE_KEY."
    )

print({
    "requested_dataset_owner": dataset_owner,
    "authentication_mode": auth_mode,
    "authenticated_username": authenticated_username,
    "authenticated_method": authenticated_method,
    "secrets_printed": False,
})


## Create and validate a tiny private Dataset


In [ ]:
import json
import subprocess
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

def run_capture(command):
    print("+", " ".join(map(str, command)), flush=True)
    result = subprocess.run(
        [str(value) for value in command],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip(), flush=True)
    return result

now = datetime.now(timezone.utc)
suffix = f"{now:%Y%m%d-%H%M%S}-{uuid.uuid4().hex[:6]}"
dataset_slug = f"typepro-owner-test-{suffix}"
if not re.fullmatch(r"[a-z0-9][a-z0-9-]{2,49}", dataset_slug):
    raise RuntimeError(f"Generated invalid Dataset slug: {dataset_slug}")
dataset_id = f"{dataset_owner}/{dataset_slug}"
title = f"TypePro owner test {now:%Y%m%d %H%M%S}"
if not 6 <= len(title) <= 50:
    raise RuntimeError(f"Generated invalid Dataset title: {title!r}")

payload_dir = Path("/kaggle/working/typepro_owner_test_payload")
payload_dir.mkdir(parents=True, exist_ok=True)
record = {
    "purpose": "Verify cross-account Kaggle Dataset ownership",
    "requested_owner": dataset_owner,
    "created_at": now.isoformat(),
    "dataset_id": dataset_id,
}
(payload_dir / "owner_test.json").write_text(
    json.dumps(record, indent=2), encoding="utf-8"
)
metadata = {
    "title": title,
    "id": dataset_id,
    "licenses": [{"name": "CC-BY-4.0"}],
}
metadata_path = payload_dir / "dataset-metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
if json.loads(metadata_path.read_text(encoding="utf-8")) != metadata:
    raise RuntimeError("dataset-metadata.json verification failed")

version_result = run_capture(["kaggle", "--version"])
if version_result.returncode:
    raise RuntimeError("Kaggle CLI is unavailable")
create_result = run_capture([
    "kaggle", "datasets", "create",
    "-p", payload_dir,
    "--dir-mode", "skip",
])
create_text = create_result.stdout or ""
create_lower = create_text.casefold()
create_succeeded = (
    "dataset is being created" in create_lower
    and "dataset creation error:" not in create_lower
)
# Kaggle CLI 2.2.4 prints a server-side Dataset creation error but can
# still exit with code 0. Never poll a slug that was not registered.
if create_result.returncode or not create_succeeded:
    raise RuntimeError(
        f"Dataset registration failed for {dataset_id}. Kaggle may have uploaded "
        "the blob without creating the Dataset. Verify Dataset write permission "
        f"for the authenticated owner. CLI output: {create_text}"
    )

# The legacy status endpoint can return 403 even immediately after a
# successful create. Listing the uploaded file is a stronger end-to-end
# verification and works with legacy credentials.
deadline = time.monotonic() + 600
final_status = None
last_files_output = ""
while time.monotonic() < deadline:
    files_result = run_capture([
        "kaggle", "datasets", "files", dataset_id, "--page-size", "10"
    ])
    last_files_output = files_result.stdout or ""
    if files_result.returncode == 0 and "owner_test.json" in last_files_output:
        final_status = "ready; owner_test.json is listable"
        break
    files_text = last_files_output.casefold()
    transient = any(
        marker in files_text
        for marker in ("403", "404", "forbidden", "not found", "could not find")
    )
    if files_result.returncode and not transient:
        raise RuntimeError(f"Cannot list Dataset files: {last_files_output}")
    print("Dataset files are not ready yet; checking again in 10 seconds", flush=True)
    time.sleep(10)
if final_status is None:
    raise TimeoutError(
        f"Dataset file did not become listable within 600 seconds: {dataset_id}; "
        f"last output: {last_files_output}"
    )

dataset_url = f"https://www.kaggle.com/datasets/{dataset_id}"
print(json.dumps({
    "verified": True,
    "owner": dataset_owner,
    "dataset_id": dataset_id,
    "dataset_url": dataset_url,
    "status": final_status,
}, indent=2, ensure_ascii=False))


A result with `"verified": true` proves that Kaggle accepted the
requested `owner/slug`, finished processing it, and exposes the uploaded
test file. Open `dataset_url` while signed into the target owner account.
